In [ ]:
# 실행 시간 측정: 단일 AEDTPLT export
import time

start_aedtplt = time.perf_counter()

# 첫 번째 variation만 추출
test_table_1_aedtplt = ParametricTable.iloc[:1]

print("=" * 70)
print(f"⏱️  AEDTPLT Export 시간 측정 (1개 Variation)")
print("=" * 70)

# AEDTPLT export 실행
aedtplt_files_test = export_aedtplt_for_variations(
    m2d_obj=m2d,
    parametric_table=test_table_1_aedtplt,
    quantity="Mag_B",
    solution="Setup1 : Transient",
    assignment="AllObjects",
    output_dir=r"D:\KDHe10\e10_example\AEDTPLT_Exports_Test",
    intrinsics={"Time": "0.06s"}
)

elapsed_aedtplt = time.perf_counter() - start_aedtplt

print(f"\n{'='*70}")
print("📊 AEDTPLT 성능 측정 결과")
print(f"{'='*70}")
print(f"✅ 총 소요 시간: {elapsed_aedtplt:.3f} 초")
print(f"📦 생성된 파일: {len(aedtplt_files_test)}개")
print(f"\n💡 전체 {len(ParametricTable)}개 variation 예상 시간:")
print(f"   약 {elapsed_aedtplt * len(ParametricTable):.1f} 초 ({elapsed_aedtplt * len(ParametricTable) / 60:.1f} 분)")
print(f"{'='*70}")

# Maxwell 2D Field Data Export

## Setup: AEDT Connection & Utilities

1. Mesh는 case로 내보내기
2. fld로 field내보내기

## 📚 패키지 정보

이 노트북은 `aedt_utils` 패키지를 사용합니다.

패키지 위치: `d:\KangDH\Emlab_emach\pyAEDT\aedt_utils\`

### 주요 기능:
- **Connection**: AEDT Desktop 자동 연결
- **Maxwell**: Maxwell 2D 디자인 자동 연결

### 사용 가능한 함수:
```python
from aedt_utils import (
    smartAedtConnector,      # 스마트 연결 (권장)
    quickConnect,            # 빠른 연결
    getDesktopConnection,    # Desktop 연결
    checkCurrentDesktopStatus, # 상태 확인
    getRunningMaxwell2d,     # Maxwell 2D 연결
)
```

import sys
sys.path.append(r'd:\KangDH\Emlab_emach\pyAEDT')

# AEDT Utils 패키지 import
from aedt_utils import (
    smartAedtConnector,
    quickConnect,
    getDesktopConnection,
    checkCurrentDesktopStatus,
    getAedtProcessesDetailed,
)

# 기본 설정
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.


## Load Maxwell 2D Model

In [ ]:
# e10 Model
AEDT_VERSION = "2025.2"
NUM_CORES = 8
NG_MODE = False  # Open AEDT UI when it is launched.
# aedt_file=r"D:\KDHe10\e10_example\e10_tutorial_ANSYSEM_2D.aedt"
# aedt_file=r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP\Design0001\e10_DOE.aedt"
# m2d = ansys.aedt.core.Maxwell2d(
#     project=aedt_file,
#     version=AEDT_VERSION,
#     new_desktop=False,
#     non_graphical=NG_MODE
# )
# Maxwell 2D 유틸리티 import
from aedt_utils import getRunningMaxwell2d

# 사용 예시
print("="*70)
print("🔌 현재 실행 중 Maxwell2d 객체 가져오기")
print("="*70)

m2d_running = getRunningMaxwell2d()

if m2d_running:
    print(f"Design Type: {m2d_running.design_type}")
    print(f"Variables: {list(m2d_running.variable_manager.variables.keys())[:5]} ...")
else:
    print("⚠️ Maxwell2d 객체를 가져오지 못했습니다.")

m2d=m2d_running


## AEDT List 추출

E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP

In [ ]:
# AEDT 파일 관리 유틸리티 import
import pandas as pd
from aedt_file_utils import find_aedt_files, find_lock_files, remove_lock_files, find, find_files

# AEDT 파일 검색 예시
# target_dir = r"E:\KDH\e10\e10_DOE\e10_DOE.opd\AMOP"
# target_dir= r"D:\KDHe10\e10_DOE\e10_DOE.opd\AMOP"
target_dir=r"C:\e10_DOE\e10_DOE.opd\AMOP"

# 재귀 검색 깊이 설정 (옵션)
# max_depth=None   → 무제한 재귀 검색 (기본값)
# max_depth=0      → 현재 디렉토리만
# max_depth=1      → 1단계 하위 디렉토리까지
# max_depth=2      → 2단계 하위 디렉토리까지
MAX_DEPTH = 1  # 원하는 깊이로 변경 가능

aedt_files = find_aedt_files(target_dir, recursive=True, max_depth=MAX_DEPTH)
lock_files = find_lock_files(target_dir, recursive=True, max_depth=MAX_DEPTH)
result = remove_lock_files(lock_files)
# print(result['message'])
import ansys.aedt.core
from concurrent.futures import ProcessPoolExecutor, as_completed
import time

# 별도 파일에서 함수 import (ProcessPoolExecutor 호환)

from aedt_parallel_processor import process_aedt_file
# DataFrame으로 변환하여 표시
if aedt_files:
    df_aedt = pd.DataFrame(aedt_files)
# Lock 파일 찾기 및 삭제 예시
file_paths = df_aedt['full_path'].tolist()
file_paths

## 범용 파일 검색 (find_files)

`find_files` 함수를 사용하여 다양한 확장자와 패턴으로 파일을 검색할 수 있습니다.

### 사용 예시:
- CSV 파일 검색
- 특정 패턴이 포함된 파일 검색
- 정규식을 사용한 고급 검색

##  AEDT 해석

여러 AEDT 파일을 병렬로 열고 Excitation 설정 및 Parametric Sweep을 자동으로 적용합니다.

##  실행

CSV 파일에서 전류와 각도 데이터를 읽어 Optimetrics 설정을 생성합니다.

- **전류**: 5 steps
- **각도**: 6 steps
- **총 Variations**: 30개 (5 × 6)

 전류와 각도 변수 추출 및 Sweep 범위 설정

 유틸리티 함수: 객체 속성 검색

`find()` 함수를 사용하여 객체의 속성이나 메서드를 쉽게 검색할 수 있습니다.

In [ ]:
125-150

In [ ]:
"""
강건한 AEDT 파일 일괄 처리 루프

기능:
1. 현재 열린 프로젝트와 처리할 파일 비교
2. Parametric Setup 존재 여부 확인
3. CSV 결과 검증 후 필요한 경우에만 실행
4. 오류 발생 시 다음 파일로 자동 이동
"""

import os
from pathlib import Path
from aedt_csv_validator import validate_and_display_csv

# 처리 결과 수집
processing_results = []

# Parametric Sweep 설정
setup_name = "ParametricSetup1"
ipeak_steps = 5
phase_steps = 6


for file_idx, fileIndex in enumerate(file_paths[125:150], 1):
    file_path = Path(fileIndex)
    file_name = file_path.name
    
    print(f"\n{'='*80}")
    print(f"[{file_idx}/{len(file_paths[125:150])}] 📁 {file_name}")
    print(f"{'='*80}")
    
    try:
        # ===== 1. 프로젝트 로드 확인 =====
        current_project_path = m2d.project_path
        target_project_path = str(file_path.absolute())
        
        print(f"\n🔍 Step 1: 프로젝트 확인")
        print(f"  - 현재 열린 프로젝트 경로: {current_project_path}")
        print(f"  - 처리할 프로젝트 경로: {target_project_path}")
        
        # 경로 비교 (대소문자 무시, 정규화)
        current_normalized = Path(current_project_path).resolve()
        target_normalized = Path(target_project_path).resolve()
        
        is_same_project = current_normalized == target_normalized
        
        # 다른 프로젝트가 열려있으면 닫고 새로 열기
        if not is_same_project:
            print(f"  ℹ️  프로젝트 전환 필요")
            print(f"  📂 현재: ...{str(current_normalized)[-50:]}")
            print(f"  📂 목표: ...{str(target_normalized)[-50:]}")
            
            try:
                m2d.close_project()
                print(f"  ✅ 이전 프로젝트 닫기 완료")
            except Exception as e:
                print(f"  ⚠️ 프로젝트 닫기 실패 (무시): {e}")
            
            print(f"  📂 프로젝트 로드 중: {file_name}")
            m2d.load_project(fileIndex)
            print(f"  ✅ 프로젝트 로드 완료")
            
            # 로드 후 경로 재확인
            loaded_path = Path(m2d.project_path).resolve()
            if loaded_path != target_normalized:
                print(f"  ⚠️ 로드된 경로가 예상과 다름:")
                print(f"    예상: {target_normalized}")
                print(f"    실제: {loaded_path}")
        else:
            print(f"  ✅ 이미 올바른 프로젝트가 열려있음")
            print(f"  📂 경로: ...{str(current_normalized)[-60:]}")
        
        # ===== 2. Excitation 설정 =====
        print(f"\n⚡ Step 2: Excitation 설정")
        try:
            excitObj = m2d.excitation_objects
            
            # 필요한 excitation 확인
            required_excitations = ['WG_Ph1_P1', 'WG_Ph2_P1', 'WG_Ph3_P1']
            missing_excitations = [name for name in required_excitations if name not in excitObj]
            
            if missing_excitations:
                print(f"  ❌ 누락된 Excitation: {missing_excitations}")
                raise KeyError(f"필수 Excitation이 없습니다: {missing_excitations}")
            
            # Excitation 객체 가져오기
            Ph1Obj = excitObj['WG_Ph1_P1']
            Ph2Obj = excitObj['WG_Ph2_P1']
            Ph3Obj = excitObj['WG_Ph3_P1']
            
            # 전류 수식 설정
            ph1Current = 'IPeak  * sin(MachineRPM/1rpm*NumPoles/60*pi*time+PhaseAdvance-0deg+0)'
            ph2Current = 'IPeak  * sin(MachineRPM/1rpm*NumPoles/60*pi*time+PhaseAdvance-120deg+0)'
            ph3Current = 'IPeak  * sin(MachineRPM/1rpm*NumPoles/60*pi*time+PhaseAdvance-240deg+0)'
            
            # 전류 적용
            Ph1Obj.update_property(prop_name='Current', prop_value=ph1Current)
            Ph2Obj.update_property(prop_name='Current', prop_value=ph2Current)
            Ph3Obj.update_property(prop_name='Current', prop_value=ph3Current)
            
            print(f"  ✅ Excitation 설정 완료 (3-phase)")
            
        except Exception as e:
            print(f"  ❌ Excitation 설정 실패: {e}")
            processing_results.append({
                'file': file_name,
                'status': '❌ Excitation 설정 실패',
                'error': str(e)
            })
            continue
        
        # ===== 3. Parametric Setup 확인 =====
        print(f"\n🔧 Step 3: Parametric Setup 확인")
        param = m2d.parametrics
        oModule = param.optimodule
        
        # 기존 Optimetrics setup 목록 확인
        existing_setups = oModule.GetChildNames()
        setup_exists = setup_name in existing_setups
        
        print(f"  - 기존 Optimetrics Setup 목록: {existing_setups}")
        print(f"  - '{setup_name}' 존재 여부: {'✅ 있음' if setup_exists else '❌ 없음'}")
        
        # ===== 4. CSV 결과 검증 =====
        print(f"\n📊 Step 4: 기존 결과 확인")
        
        # CSV 파일 경로 생성
        current_aedt_path = m2d.project_path
        aedt_dir = Path(current_aedt_path).parent
        aedt_filename = Path(current_aedt_path).stem
        csv_filename = f"{aedt_filename}_{setup_name}_Result.csv"
        csv_path = aedt_dir / csv_filename
        
        # 결과 검증 (verbose=False로 간단하게)
        validation_result = validate_and_display_csv(
            m2d_obj=m2d,
            setup_name=setup_name,
            expected_ipeak_steps=ipeak_steps,
            expected_phase_steps=phase_steps,
            verbose=False
        )
        
        results_complete = validation_result['is_complete']
        csv_exists = validation_result['csv_exists']
        
        print(f"  - CSV 파일: {'✅ 존재' if csv_exists else '❌ 없음'}")
        if csv_exists:
            print(f"  - 결과 완성도: {validation_result['actual_count']}/{validation_result['expected_count']} "
                  f"({validation_result['completion_rate']:.1f}%)")
        
        # ===== 5. 처리 결정 로직 =====
        print(f"\n🎯 Step 5: 처리 결정")
        
        if results_complete:
            print(f"  ✅ 결과가 이미 완전함 - Sweep 생략")
            processing_results.append({
                'file': file_name,
                'status': '✅ Already Complete',
                'completion_rate': validation_result['completion_rate']
            })
            continue
        
        # 예상 Sweep 설정 정의
        expected_sweep_config = {
            "IPeak": {"Data": "LINC 10A 650.53A 5", "steps": ipeak_steps},
            "PhaseAdvance": {"Data": "LINC 0deg 90deg 6", "steps": phase_steps}
        }
        
        # Setup이 없으면 생성
        if not setup_exists:
            print(f"  📝 Parametric Setup 생성 중...")
            try:
                oModule.InsertSetup("OptiParametric", 
                    [
                        "NAME:ParametricSetup1",
                        "IsEnabled:=", True,
                        [
                            "NAME:ProdOptiSetupDataV2",
                            "SaveFields:=", True,
                            "CopyMesh:=", False,
                            "SolveWithCopiedMeshOnly:=", False
                        ],
                        "InterpolationPoints:=", 0,
                        ["NAME:StartingPoint"],
                        "Sim. Setups:=", ["Setup1"],
                        [
                            "NAME:Sweeps",
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "IPeak",
                                "Data:=", "LINC 10A 650.53A 5",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ],
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "PhaseAdvance",
                                "Data:=", "LINC 0deg 90deg 6",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ]
                        ],
                        ["NAME:Sweep Operations"],
                        ["NAME:Goals"]
                    ])
                print(f"  ✅ Parametric Setup 생성 완료")
            except Exception as e:
                print(f"  ❌ Parametric Setup 생성 실패: {e}")
                processing_results.append({
                    'file': file_name,
                    'status': '❌ Setup 생성 실패',
                    'error': str(e)
                })
                continue
        else:
            print(f"  ℹ️  Parametric Setup이 이미 존재함")
            
            # ===== 5-1. 기존 Setup 설정 검증 =====
            print(f"\n🔍 Step 5-1: Parametric Setup 설정 검증")
            try:
                cursimul = oModule.GetChildObject(setup_name)
                
                # Sweep 변수 확인
                sweep_vars = cursimul.GetSweepVariables()
                print(f"  - Sweep 변수: {sweep_vars}")
                
                # 설정 불일치 플래그
                config_mismatch = False
                
                # 예상 변수 확인
                expected_vars = list(expected_sweep_config.keys())
                if set(sweep_vars) != set(expected_vars):
                    print(f"  ⚠️ Sweep 변수 불일치:")
                    print(f"    예상: {expected_vars}")
                    print(f"    실제: {sweep_vars}")
                    config_mismatch = True
                else:
                    print(f"  ✅ Sweep 변수 일치: {sweep_vars}")
                    
                    # 각 변수의 범위 확인
                    for var in sweep_vars:
                        try:
                            var_data = cursimul.GetSweepData(var)
                            expected_data = expected_sweep_config[var]["Data"]
                            
                            # 간단한 문자열 비교 (공백 제거)
                            var_data_normalized = var_data.replace(" ", "").upper()
                            expected_data_normalized = expected_data.replace(" ", "").upper()
                            
                            if var_data_normalized != expected_data_normalized:
                                print(f"  ⚠️ {var} 범위 불일치:")
                                print(f"    예상: {expected_data}")
                                print(f"    실제: {var_data}")
                                config_mismatch = True
                            else:
                                print(f"  ✅ {var} 범위 일치: {var_data}")
                        except Exception as e:
                            print(f"  ⚠️ {var} 범위 확인 실패: {e}")
                            config_mismatch = True
                
                # 설정이 일치하지 않으면 Setup 삭제 후 재생성
                if config_mismatch:
                    print(f"\n  ⚠️ Setup 설정 불일치 - 재생성 필요")
                    print(f"  🗑️  기존 Setup 삭제 중...")
                    try:
                        oModule.DeleteSetups([setup_name])
                        print(f"  ✅ 기존 Setup 삭제 완료")
                        
                        print(f"  📝 Parametric Setup 재생성 중...")
                        oModule.InsertSetup("OptiParametric", 
                            [
                                "NAME:ParametricSetup1",
                                "IsEnabled:=", True,
                                [
                                    "NAME:ProdOptiSetupDataV2",
                                    "SaveFields:=", True,
                                    "CopyMesh:=", False,
                                    "SolveWithCopiedMeshOnly:=", False
                                ],
                                "InterpolationPoints:=", 0,
                                ["NAME:StartingPoint"],
                                "Sim. Setups:=", ["Setup1"],
                                [
                                    "NAME:Sweeps",
                                    [
                                        "NAME:SweepDefinition",
                                        "Variable:=", "IPeak",
                                        "Data:=", "LINC 10A 650.53A 5",
                                        "OffsetF1:=", False,
                                        "Synchronize:=", 0
                                    ],
                                    [
                                        "NAME:SweepDefinition",
                                        "Variable:=", "PhaseAdvance",
                                        "Data:=", "LINC 0deg 90deg 6",
                                        "OffsetF1:=", False,
                                        "Synchronize:=", 0
                                    ]
                                ],
                                ["NAME:Sweep Operations"],
                                ["NAME:Goals"]
                            ])
                        print(f"  ✅ Parametric Setup 재생성 완료")
                    except Exception as e:
                        print(f"  ❌ Setup 재생성 실패: {e}")
                        processing_results.append({
                            'file': file_name,
                            'status': '❌ Setup 재생성 실패',
                            'error': str(e)
                        })
                        continue
                else:
                    print(f"  ✅ Setup 설정 검증 통과")
                    
            except Exception as e:
                print(f"  ⚠️ Setup 설정 검증 실패: {e}")
                print(f"  ℹ️  기존 Setup 그대로 사용")
        
        # ===== 6. Parametric Sweep 실행 =====
        print(f"\n⚙️  Step 6: Parametric Sweep 실행 결정")
        
        # CSV가 불완전하면 Setup 삭제 후 재실행
        if csv_exists and not results_complete:
            print(f"  ⚠️ 기존 결과가 불완전함 ({validation_result['actual_count']}/{validation_result['expected_count']})")
            print(f"  🗑️  기존 Setup 삭제 후 재실행")
            
            try:
                # Setup 삭제
                oModule.DeleteSetups([setup_name])
                print(f"  ✅ 기존 Setup 삭제 완료")
                
                # Setup 재생성
                print(f"  📝 Parametric Setup 재생성 중...")
                oModule.InsertSetup("OptiParametric", 
                    [
                        "NAME:ParametricSetup1",
                        "IsEnabled:=", True,
                        [
                            "NAME:ProdOptiSetupDataV2",
                            "SaveFields:=", True,
                            "CopyMesh:=", False,
                            "SolveWithCopiedMeshOnly:=", False
                        ],
                        "InterpolationPoints:=", 0,
                        ["NAME:StartingPoint"],
                        "Sim. Setups:=", ["Setup1"],
                        [
                            "NAME:Sweeps",
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "IPeak",
                                "Data:=", "LINC 10A 650.53A 5",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ],
                            [
                                "NAME:SweepDefinition",
                                "Variable:=", "PhaseAdvance",
                                "Data:=", "LINC 0deg 90deg 6",
                                "OffsetF1:=", False,
                                "Synchronize:=", 0
                            ]
                        ],
                        ["NAME:Sweep Operations"],
                        ["NAME:Goals"]
                    ])
                print(f"  ✅ Parametric Setup 재생성 완료")
                
                # CSV 파일도 삭제 (불완전한 결과 제거)
                if csv_path.exists():
                    csv_path.unlink()
                    print(f"  🗑️  불완전한 CSV 파일 삭제: {csv_filename}")
                
            except Exception as e:
                print(f"  ❌ Setup 재생성 실패: {e}")
                processing_results.append({
                    'file': file_name,
                    'status': '❌ Setup 재생성 실패 (불완전)',
                    'error': str(e)
                })
                continue
        
        # CSV가 없거나 재생성한 경우에만 실행
        if not csv_exists or (csv_exists and not results_complete):
            try:
                print(f"  🔄 Sweep 실행 중... (예상 시간: 수십 분 이상)")
                cursimul = oModule.GetChildObject(setup_name)
                cursimul.StartAnalyze()
                print(f"  ✅ Sweep 실행 완료")
                
                # ===== 7. CSV Export =====
                print(f"\n💾 Step 7: 결과 Export")
                try:
                    oModule.ExportOptimetricsResult(setup_name, str(csv_path), False)
                    print(f"  ✅ CSV Export 완료: {csv_filename}")
                    
                    # 최종 검증
                    final_validation = validate_and_display_csv(
                        m2d_obj=m2d,
                        setup_name=setup_name,
                        expected_ipeak_steps=ipeak_steps,
                        expected_phase_steps=phase_steps,
                        verbose=False
                    )
                    
                    processing_results.append({
                        'file': file_name,
                        'status': '✅ Sweep 완료' if final_validation['is_complete'] else '⚠️ Sweep 완료 (불완전)',
                        'completion_rate': final_validation['completion_rate']
                    })
                    
                except Exception as e:
                    print(f"  ⚠️ CSV Export 실패: {e}")
                    processing_results.append({
                        'file': file_name,
                        'status': '⚠️ Sweep 완료 (Export 실패)',
                        'error': str(e)
                    })
                    
            except Exception as e:
                print(f"  ❌ Sweep 실행 실패: {e}")
                processing_results.append({
                    'file': file_name,
                    'status': '❌ Sweep 실행 실패',
                    'error': str(e)
                })
                continue
        else:
            print(f"  ℹ️  기존 결과가 완전함 - Sweep 생략")
            processing_results.append({
                'file': file_name,
                'status': '✅ 기존 결과 사용',
                'completion_rate': validation_result['completion_rate']
            })
    
    except Exception as e:
        print(f"\n❌ 파일 처리 중 예외 발생: {e}")
        import traceback
        traceback.print_exc()
        
        processing_results.append({
            'file': file_name,
            'status': '❌ 처리 실패',
            'error': str(e)
        })
        continue

# ===== 최종 결과 요약 =====
print("\n" + "=" * 80)
print("📊 처리 결과 요약")
print("=" * 80)

if processing_results:
    import pandas as pd
    df_results = pd.DataFrame(processing_results)
    display(df_results)
    
    print(f"\n✅ 성공: {sum('✅' in r['status'] for r in processing_results)}개")
    print(f"⚠️ 경고: {sum('⚠️' in r['status'] for r in processing_results)}개")
    print(f"❌ 실패: {sum('❌' in r['status'] for r in processing_results)}개")
else:
    print("ℹ️  처리된 파일이 없습니다.")

print("=" * 80)

In [ ]:
# Parametric Sweep 결과 검증 (validate_and_display_csv 함수 사용)
from aedt_csv_validator import validate_and_display_csv

# Parametric Sweep 설정
# IPeak: 5 steps (10A ~ 650.53A)
# PhaseAdvance: 6 steps (0deg ~ 90deg)
ipeak_steps = 5
phase_steps = 6

# 검증 실행 (Maxwell2d 객체로부터 자동으로 경로 추출)
validation_result = validate_and_display_csv(
    m2d_obj=m2d,
    setup_name="ParametricSetup1",
    expected_ipeak_steps=ipeak_steps,
    expected_phase_steps=phase_steps,
    verbose=True
)

## FLD Export Function Definition

In [ ]:
m2dpost=m2d.post
all_objects = m2d.modeler.object_names

# Modelplotter=m2dpost.get_model_plotter_geometries(generate_mesh=True,get_objects_from_aedt=True)
### mesh export as *.case 
import os 
desktop = ansys.aedt.core.Desktop() 
pjtPath=desktop.project_path()
prjName=desktop.active_project().GetName()
filePath=os.path.join(pjtPath, prjName) 
pjt=desktop.load_project(filePath)
setup=pjt.get_setup(name='Setup1')
FieldReporter=pjt.get_module("FieldsReporter")

SetupObj=m2d.get_setup('Setup1')
{"Time":SetupObj.props['MaxTimeStep']}
import ansys.aedt.core.visualization.plot.pyvista as AEDTvista
AEDTvista
import ansys.aedt.core.visualization.post as AEDTpost
AEDTpost

## plot Field

# AEDTPLT Export: Utility Functions

이 섹션은 AEDT 파일별, Sweep별, Time별로 `.aedtplt` 파일을 추출하는 유틸리티 함수들을 제공합니다.

### 주요 함수:
1. `get_parametric_sweep_table()`: Parametric Sweep 결과에서 IPeak, PhaseAdvance 값 추출
2. `get_time_steps()`: Transient 해석 결과에서 time step 목록 추출
3. `export_aedtplt_batch()`: 일괄 AEDTPLT 파일 export

#### get_parametric_sweep_table

In [ ]:
import pandas as pd
import os
from pathlib import Path

def get_parametric_sweep_table(m2d_obj, setup_name=None, auto_export=True):
    """
    Parametric Sweep 결과에서 IPeak, PhaseAdvance 값을 추출합니다.
    CSV가 없으면 자동으로 Optimetrics에서 Export합니다.
    
    Parameters:
    -----------
    m2d_obj : Maxwell2d object
        AEDT Maxwell 2D 객체
    setup_name : str, optional
        Parametric Setup 이름 (None이면 자동 감지)
    auto_export : bool
        CSV가 없을 때 자동으로 Export 수행 여부 (default: True)
    
    Returns:
    --------
    pd.DataFrame : IPeak, PhaseAdvance 값을 포함한 DataFrame
    """
    try:
        # Parametrics 모듈 가져오기
        param = m2d_obj.parametrics
        oModule = param.optimodule
        
        # Setup 이름 자동 감지
        existing_setups = oModule.GetChildNames()
        
        if setup_name is None:
            # 자동 감지: OptiParametric 타입의 Setup 찾기
            if not existing_setups:
                print(f"  ❌ Parametric Setup이 존재하지 않습니다.")
                return None
            
            # 첫 번째 Setup 사용 (보통 ParametricSetup1)
            setup_name = existing_setups[0]
            print(f"  🔍 Parametric Setup 자동 감지: '{setup_name}'")
        else:
            # Setup 존재 확인
            if setup_name not in existing_setups:
                print(f"  ❌ Parametric Setup '{setup_name}'이 존재하지 않습니다.")
                print(f"  📋 존재하는 Setup: {existing_setups}")
                return None
        
        # CSV 파일 경로 생성
        aedt_path = Path(m2d_obj.project_path)
        aedt_dir = aedt_path.parent
        aedt_filename = aedt_path.stem
        csv_filename = f"{aedt_filename}_{setup_name}_Result.csv"
        csv_path = aedt_dir / csv_filename
        
        # CSV 파일이 있으면 읽기
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            
            # IPeak, PhaseAdvance 컬럼이 있는지 확인
            required_cols = ['IPeak', 'PhaseAdvance']
            if all(col in df.columns for col in required_cols):
                print(f"  ✅ CSV 파일에서 {len(df)}개 행 읽기 완료")
                return df[required_cols].drop_duplicates().reset_index(drop=True)
            else:
                print(f"  ⚠️ CSV 파일에 필요한 컬럼이 없습니다: {required_cols}")
                print(f"  ℹ️  CSV 재생성 시도...")
                # CSV 삭제 후 재생성
                csv_path.unlink()
        
        # CSV 파일이 없거나 컬럼이 없는 경우
        if not auto_export:
            print(f"  ⚠️ CSV 파일이 없고 auto_export=False입니다.")
            return None
        
        print(f"  📤 Optimetrics 결과 Export 시도 중...")
        
        # CSV Export 실행
        try:
            print(f"  💾 CSV Export 실행: {csv_filename}")
            oModule.ExportOptimetricsResult(setup_name, str(csv_path), False)
            print(f"  ✅ CSV Export 완료")
            
            # Export된 CSV 읽기
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                
                # IPeak, PhaseAdvance 컬럼 확인
                required_cols = ['IPeak', 'PhaseAdvance']
                if all(col in df.columns for col in required_cols):
                    print(f"  ✅ CSV에서 {len(df)}개 행 읽기 완료")
                    return df[required_cols].drop_duplicates().reset_index(drop=True)
                else:
                    print(f"  ⚠️ Export된 CSV에 필요한 컬럼이 없습니다: {required_cols}")
                    print(f"  📋 실제 컬럼: {list(df.columns)}")
                    return None
            else:
                print(f"  ❌ CSV Export 후에도 파일이 없습니다: {csv_path}")
                return None
                
        except Exception as e_export:
            print(f"  ❌ CSV Export 실패: {e_export}")
            return None
            
    except Exception as e:
        print(f"  ❌ Parametric Table 추출 실패: {e}")
        import traceback
        traceback.print_exc()
        return None




#### get_time_steps

In [ ]:

def get_time_steps(m2d_obj, setup_name="Setup1"):
    """
    Transient 해석 결과에서 time step 목록을 추출합니다.
    
    Parameters:
    -----------
    m2d_obj : Maxwell2d object
        AEDT Maxwell 2D 객체
    setup_name : str
        Setup 이름 (default: "Setup1")
    
    Returns:
    --------
    list : time step 값 목록 (단위 포함, 예: "0.00011494252873563217s")
    """
    import re
    
    # 방법 1: post 객체를 통한 실제 솔루션 데이터에서 추출 (가장 정확함)
    try:
        print(f"  🔄 Post 객체를 통한 time step 추출 시도...")
        
        # Solution data 가져오기
        time_data = m2d_obj.post.get_solution_data_per_variation(
            setup_name=setup_name,
            domain="Time",
            expressions=["Time"]
        )
        
        if time_data:
            # intrinsics에서 Time 추출
            for data in time_data:
                if hasattr(data, 'intrinsics') and 'Time' in data.intrinsics:
                    time_values = data.intrinsics['Time']
                    print(f"  ✅ Post 객체에서 {len(time_values)}개 time step 추출 (실제 솔루션 데이터)")
                    print(f"  📝 첫 3개: {time_values[:3]}")
                    print(f"  📝 마지막: {time_values[-1]}")
                    return time_values
                    
    except Exception as e_post:
        print(f"  ⚠️ Post 객체 추출 실패: {e_post}")
    
    # 방법 2: Setup.properties + Variable 평가를 통한 추출
    try:
        setup_obj = m2d_obj.get_setup(setup_name)
        prop_obj = setup_obj.properties
        
        print(f"  🔍 Setup properties 확인 중...")
        
        # Time Step과 Stop Time 확인
        time_step_expr = None
        stop_time_expr = None
        
        if 'Time Step' in prop_obj:
            time_step_expr = prop_obj['Time Step']
            print(f"  📊 Time Step Expression: {time_step_expr}")
        
        for key in ['Stop Time', 'StopTime', 'Stop time']:
            if key in prop_obj:
                stop_time_expr = prop_obj[key]
                print(f"  📊 Stop Time Expression: {stop_time_expr}")
                break
        
        # Expression이 변수를 포함하는지 확인
        if time_step_expr and stop_time_expr:
            # Design 변수 가져오기
            design_vars = m2d_obj.variable_manager.variables
            
            print(f"  🔧 Design 변수 확인 중...")
            
            # 주요 변수 값 출력
            key_vars = ['NumTorquePointsPerCycle', 'NumTorqueCycles', 'MachineRPM', 'NumPoles']
            var_dict = {}
            
            for var_name in key_vars:
                if var_name in design_vars:
                    var_value = design_vars[var_name].value
                    var_dict[var_name] = var_value
                    print(f"    • {var_name}: {var_value}")
            
            # Time Step Expression 평가
            time_step_evaluated = None
            stop_time_evaluated = None
            
            # Expression에 변수가 포함되어 있는지 확인
            if any(char.isalpha() for char in str(time_step_expr)):
                print(f"  📐 Time Step expression 계산 중...")
                
                try:
                    # AEDT의 evaluate 기능 우선 시도
                    oDesign = m2d_obj.odesign
                    time_step_result = oDesign.GetNominalVariation().EvaluateVariationValue(
                        time_step_expr
                    )
                    
                    # 결과에서 숫자 추출
                    time_step_match = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(time_step_result))
                    if time_step_match:
                        time_step_evaluated = float(time_step_match[0])
                        print(f"  ✅ Time Step 계산 결과: {time_step_evaluated}s")
                    
                except Exception as e1:
                    print(f"  ⚠️ EvaluateVariationValue 실패, 수동 계산 시도: {e1}")
                    
                    # 수동으로 변수 치환 및 계산
                    try:
                        expr_for_eval = str(time_step_expr)
                        
                        # 변수 치환 (design_vars[var_name].value 사용)
                        for var_name in key_vars:
                            if var_name in expr_for_eval and var_name in design_vars:
                                var_value_str = str(design_vars[var_name].value)
                                # 숫자 부분만 추출 (단위 제거)
                                var_num = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", var_value_str)
                                if var_num:
                                    expr_for_eval = expr_for_eval.replace(var_name, var_num[0])
                        
                        # 단위 제거
                        expr_for_eval = re.sub(r'/1rpm', '', expr_for_eval)
                        expr_for_eval = re.sub(r'[a-zA-Z]+', '', expr_for_eval)
                        
                        # 계산
                        time_step_evaluated = eval(expr_for_eval)
                        print(f"  ✅ 수동 계산 Time Step: {time_step_evaluated}s")
                        
                    except Exception as e2:
                        print(f"  ⚠️ 수동 계산 실패: {e2}")
            else:
                # 숫자만 있는 경우
                time_step_match = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(time_step_expr))
                if time_step_match:
                    time_step_evaluated = float(time_step_match[0])
                    print(f"  ✅ Time Step (직접 값): {time_step_evaluated}s")
            
            # Stop Time Expression 평가
            if any(char.isalpha() for char in str(stop_time_expr)):
                print(f"  📐 Stop Time expression 계산 중...")
                
                try:
                    oDesign = m2d_obj.odesign
                    stop_time_result = oDesign.GetNominalVariation().EvaluateVariationValue(
                        stop_time_expr
                    )
                    
                    # 결과에서 숫자 추출
                    stop_time_match = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(stop_time_result))
                    if stop_time_match:
                        stop_time_evaluated = float(stop_time_match[0])
                        print(f"  ✅ Stop Time 계산 결과: {stop_time_evaluated}s")
                    
                except Exception as e3:
                    print(f"  ⚠️ EvaluateVariationValue 실패, 수동 계산 시도: {e3}")
                    
                    try:
                        expr_for_eval = str(stop_time_expr)
                        
                        # 변수 치환
                        for var_name in key_vars:
                            if var_name in expr_for_eval and var_name in design_vars:
                                var_value_str = str(design_vars[var_name].value)
                                var_num = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", var_value_str)
                                if var_num:
                                    expr_for_eval = expr_for_eval.replace(var_name, var_num[0])
                        
                        # 단위 제거
                        expr_for_eval = re.sub(r'/1rpm', '', expr_for_eval)
                        expr_for_eval = re.sub(r'[a-zA-Z]+', '', expr_for_eval)
                        
                        stop_time_evaluated = eval(expr_for_eval)
                        print(f"  ✅ 수동 계산 Stop Time: {stop_time_evaluated}s")
                        
                    except Exception as e4:
                        print(f"  ⚠️ 수동 계산 실패: {e4}")
            else:
                stop_time_match = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(stop_time_expr))
                if stop_time_match:
                    stop_time_evaluated = float(stop_time_match[0])
                    print(f"  ✅ Stop Time (직접 값): {stop_time_evaluated}s")
            
            # Time step 목록 생성
            if time_step_evaluated and stop_time_evaluated:
                num_steps = int(stop_time_evaluated / time_step_evaluated) + 1
                time_steps = [f"{i * time_step_evaluated}s" for i in range(num_steps)]
                
                print(f"  ✅ {len(time_steps)}개 time step 생성 (계산값)")
                print(f"  📝 첫 3개: {time_steps[:3]}")
                print(f"  📝 마지막: {time_steps[-1]}")
                
                return time_steps
        
        print(f"  ℹ️  Time Step 또는 Stop Time 평가 실패")
        
    except Exception as e:
        print(f"  ⚠️ Setup.properties 추출 실패: {e}")
        import traceback
        traceback.print_exc()
    
    # 방법 3: Setup.props를 사용한 추출 (기존 방식)
    try:
        print(f"  🔄 Setup.props를 통한 추출 시도...")
        setup_obj = m2d_obj.get_setup(setup_name)
        
        max_time_step = setup_obj.props.get('MaxTimeStep', '0.0001s')
        stop_time = setup_obj.props.get('StopTime', '0.06s')
        
        print(f"  📊 MaxTimeStep={max_time_step}, StopTime={stop_time}")
        
        # 문자열에서 숫자와 단위 분리
        max_step_match = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", max_time_step)
        stop_time_match = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", stop_time)
        time_unit_match = re.findall(r"[a-zA-Z]+", max_time_step)
        
        if max_step_match and stop_time_match and time_unit_match:
            max_step_val = float(max_step_match[0])
            stop_time_val = float(stop_time_match[0])
            time_unit = time_unit_match[0]
            
            # Time step 목록 생성
            num_steps = int(stop_time_val / max_step_val) + 1
            time_steps = [f"{i * max_step_val}{time_unit}" for i in range(num_steps)]
            
            print(f"  ✅ Setup.props 기반으로 {len(time_steps)}개 time step 생성")
            print(f"  📝 첫 번째: {time_steps[0]}, 마지막: {time_steps[-1]}")
            
            return time_steps
            
    except Exception as e:
        print(f"  ⚠️ Setup.props 추출 실패: {e}")
    
    print(f"  ❌ 모든 방법으로 time step 추출 실패")
    return []



#### export_aedtplt_batch

In [9]:

def export_aedtplt_batch(
    m2d_obj,
    output_base_dir,
    file_paths,
    setup_name="Setup1",
    parametric_setup_name="ParametricSetup1",
    quantity="A_Vector",
    plot_name="A_Vector1",
    create_plot_if_missing=True
):
    """
    여러 AEDT 파일에 대해 Sweep별, Time별로 AEDTPLT 파일을 일괄 export합니다.
    
    Parameters:
    -----------
    m2d_obj : Maxwell2d object
        AEDT Maxwell 2D 객체
    output_base_dir : str
        Export 파일을 저장할 기본 디렉토리
    file_paths : list
        처리할 AEDT 파일 경로 목록
    setup_name : str
        Setup 이름 (default: "Setup1")
    parametric_setup_name : str
        Parametric Setup 이름 (default: "ParametricSetup1")
    quantity : str
        Export할 필드 물리량 (default: "A_Vector")
    plot_name : str
        Field plot 이름 (default: "A_Vector1")
    create_plot_if_missing : bool
        Plot이 없을 경우 자동 생성 여부 (default: True)
    
    Returns:
    --------
    dict : 처리 결과 통계
    """
    
    results = {
        'total_files': len(file_paths),
        'processed_files': 0,
        'failed_files': 0,
        'total_exported': 0,
        'details': []
    }
    
    for file_idx, file_path in enumerate(file_paths, 1):
        file_name = Path(file_path).name
        
        print(f"\n{'='*80}")
        print(f"[{file_idx}/{len(file_paths)}] 📁 {file_name}")
        print(f"{'='*80}")
        
        try:
            # ===== 1. 프로젝트 로드 =====
            current_project_path = Path(m2d_obj.project_path).resolve()
            target_project_path = Path(file_path).resolve()
            
            if current_project_path != target_project_path:
                print(f"  📂 프로젝트 전환 중...")
                try:
                    m2d_obj.close_project()
                except:
                    pass
                m2d_obj.load_project(str(file_path))
                print(f"  ✅ 프로젝트 로드 완료")
            else:
                print(f"  ✅ 이미 올바른 프로젝트가 열려있음")
            
            # ===== 2. Parametric Table 추출 =====
            print(f"\n📊 Parametric Sweep 데이터 추출...")
            parametric_table = get_parametric_sweep_table(m2d_obj, parametric_setup_name)
            
            if parametric_table is None or parametric_table.empty:
                print(f"  ⚠️ Parametric Table이 없음 - Skip")
                results['failed_files'] += 1
                results['details'].append({
                    'file': file_name,
                    'status': '⚠️ No Parametric Data',
                    'exported': 0
                })
                continue
            
            print(f"  ✅ {len(parametric_table)}개 Variation 발견")
            
            # ===== 3. Time Steps 추출 =====
            print(f"\n⏱️  Time Step 추출...")
            time_steps = get_time_steps(m2d_obj, setup_name)
            
            if not time_steps:
                print(f"  ⚠️ Time Step이 없음 - Skip")
                results['failed_files'] += 1
                results['details'].append({
                    'file': file_name,
                    'status': '⚠️ No Time Steps',
                    'exported': 0
                })
                continue
            
            print(f"  ✅ {len(time_steps)}개 Time Step 발견")
            
            # ===== 4. Field Plot 확인/생성 =====
            print(f"\n🎨 Field Plot 확인...")
            oDesign = m2d_obj.odesign
            oModule = oDesign.GetModule("FieldsReporter")
            
            # 기존 Plot 목록 확인
            existing_plots = oModule.GetFieldPlotNames()
            
            if plot_name not in existing_plots:
                if create_plot_if_missing:
                    print(f"  📝 Field Plot 생성: {plot_name}")
                    try:
                        # 모든 오브젝트 이름 가져오기
                        all_object_names = m2d_obj.modeler.object_names
                        num_objects = len(all_object_names)
                        
                        # PlotGeomInfo 구성: [Type, "Surface", "FacesList", NumObjects, obj1, obj2, ...]
                        plot_geom_info = [1, "Surface", "FacesList", num_objects] + all_object_names
                        
                        # A_Vector1 생성
                        oModule.CreateFieldPlot(
                            [
                                "NAME:" + plot_name,
                                "SolutionName:=", f"{setup_name} : Transient",
                                "UserSpecifyName:=", 1,
                                "UserSpecifyFolder:=", 1,
                                "QuantityName:=", quantity,
                                "PlotFolder:=", "A",
                                "StreamlinePlot:=", False,
                                "AdjacentSidePlot:=", False,
                                "FullModelPlot:=", False,
                                "IntrinsicVar:=", "Time='0s'",
                                "PlotGeomInfo:=", plot_geom_info,
                                "FilterBoxes:=", [0],
                                [
                                    "NAME:PlotOnSurfaceSettings",
                                    "ShadingType:=", 0,
                                    "Filled:=", False,
                                    "IsoValType:=", "Tone",
                                    "AddGrid:=", False,
                                    "MapTransparency:=", True,
                                    "Refinement:=", 0,
                                    "Transparency:=", 0,
                                    "SmoothingLevel:=", 0,
                                    [
                                        "NAME:Arrow3DSpacingSettings",
                                        "ArrowUniform:=", True,
                                        "ArrowSpacing:=", 0,
                                        "MinArrowSpacing:=", 0,
                                        "MaxArrowSpacing:=", 0
                                    ],
                                    "GridColor:=", [255, 255, 255]
                                ],
                                "EnableGaussianSmoothing:=", False,
                                "SurfaceOnly:=", False
                            ],
                            "Field"
                        )
                        print(f"  ✅ Field Plot 생성 완료 ({num_objects}개 오브젝트)")
                    except Exception as e:
                        print(f"  ⚠️ Field Plot 생성 실패: {e}")
                        print(f"  ℹ️  기존 Plot 사용 시도")
                        if existing_plots:
                            plot_name = existing_plots[0]
                            print(f"  📌 사용할 Plot: {plot_name}")
                else:
                    print(f"  ⚠️ Field Plot이 없음 - Skip")
                    results['failed_files'] += 1
                    results['details'].append({
                        'file': file_name,
                        'status': '⚠️ No Field Plot',
                        'exported': 0
                    })
                    continue
            else:
                print(f"  ✅ Field Plot 존재: {plot_name}")
            
            # ===== 5. Export 디렉토리 생성 =====
            aedt_stem = Path(file_path).stem
            export_dir = Path(output_base_dir) / aedt_stem
            export_dir.mkdir(parents=True, exist_ok=True)
            
            print(f"\n💾 Export 디렉토리: {export_dir}")
            
            # ===== 6. AEDTPLT Export =====
            print(f"\n🚀 AEDTPLT Export 시작...")
            print(f"  - Total: {len(parametric_table)} variations × {len(time_steps)} time steps")
            print(f"  - Expected files: {len(parametric_table) * len(time_steps)}")
            
            exported_count = 0
            
            for var_idx, row in parametric_table.iterrows():
                ipeak_val = row['IPeak']
                phase_val = row['PhaseAdvance']
                
                # IPeak, PhaseAdvance 설정
                try:
                    oDesign.ChangeProperty(
                        [
                            "NAME:AllTabs",
                            [
                                "NAME:LocalVariableTab",
                                [
                                    "NAME:PropServers", 
                                    "LocalVariables"
                                ],
                                [
                                    "NAME:ChangedProps",
                                    [
                                        "NAME:IPeak",
                                        "Value:=", str(ipeak_val)
                                    ],
                                    [
                                        "NAME:PhaseAdvance",
                                        "Value:=", str(phase_val)
                                    ]
                                ]
                            ]
                        ])
                except Exception as e:
                    print(f"  ⚠️ Variable 설정 실패 (IPeak={ipeak_val}, Phase={phase_val}): {e}")
                    continue
                
                # 각 Time Step에 대해 Export
                for time_idx, time_value in enumerate(time_steps):
                    try:
                        # Time 문자열 생성 (예: "Time='0.00011494252873563217s'")
                        time_str = f"Time='{time_value}'"
                        
                        # 파일명 생성
                        # 예: e10_DOE_IPeak170.13A_Phase18deg_Time000.aedtplt
                        file_name_export = (
                            f"{aedt_stem}_"
                            f"IPeak{ipeak_val}_"
                            f"Phase{phase_val}_"
                            f"Time{time_idx:03d}.aedtplt"
                        )
                        file_path_export = export_dir / file_name_export
                        
                        # Solution context 설정 (Plot을 특정 시간으로 설정)
                        oModule.SetPlotsSolutionContext(
                            [plot_name], 
                            f"{setup_name} : Transient", 
                            time_str
                        )
                        
                        # Plot 업데이트 (여러 방법 시도)
                        try:
                            # 방법 1: 모든 field plot 업데이트
                            oModule.UpdateAllFieldPlots()
                        except:
                            try:
                                # 방법 2: Plot을 삭제하고 재생성
                                oModule.DeleteFieldPlot([plot_name])
                                
                                all_object_names = m2d_obj.modeler.object_names
                                num_objects = len(all_object_names)
                                plot_geom_info = [1, "Surface", "FacesList", num_objects] + all_object_names
                                
                                oModule.CreateFieldPlot(
                                    [
                                        "NAME:" + plot_name,
                                        "SolutionName:=", f"{setup_name} : Transient",
                                        "UserSpecifyName:=", 1,
                                        "UserSpecifyFolder:=", 1,
                                        "QuantityName:=", quantity,
                                        "PlotFolder:=", "A",
                                        "StreamlinePlot:=", False,
                                        "AdjacentSidePlot:=", False,
                                        "FullModelPlot:=", False,
                                        "IntrinsicVar:=", time_str,
                                        "PlotGeomInfo:=", plot_geom_info,
                                        "FilterBoxes:=", [0],
                                        [
                                            "NAME:PlotOnSurfaceSettings",
                                            "ShadingType:=", 0,
                                            "Filled:=", False,
                                            "IsoValType:=", "Tone",
                                            "AddGrid:=", False,
                                            "MapTransparency:=", True,
                                            "Refinement:=", 0,
                                            "Transparency:=", 0,
                                            "SmoothingLevel:=", 0,
                                            [
                                                "NAME:Arrow3DSpacingSettings",
                                                "ArrowUniform:=", True,
                                                "ArrowSpacing:=", 0,
                                                "MinArrowSpacing:=", 0,
                                                "MaxArrowSpacing:=", 0
                                            ],
                                            "GridColor:=", [255, 255, 255]
                                        ],
                                        "EnableGaussianSmoothing:=", False,
                                        "SurfaceOnly:=", False
                                    ],
                                    "Field"
                                )
                            except Exception as e2:
                                print(f"    ⚠️ Plot 업데이트/재생성 실패: {e2}")
                        
                        # Field plot export
                        oModule.ExportFieldPlot(plot_name, False, str(file_path_export))
                        
                        exported_count += 1
                        
                        if exported_count % 10 == 0:
                            print(f"  📦 [{exported_count}/{len(parametric_table) * len(time_steps)}] Exported...")
                        
                    except Exception as e:
                        print(f"  ❌ Export 실패 (Var{var_idx+1}, Time{time_idx}): {e}")
            
            print(f"\n✅ Export 완료: {exported_count}개 파일")
            
            results['processed_files'] += 1
            results['total_exported'] += exported_count
            results['details'].append({
                'file': file_name,
                'status': '✅ Success',
                'exported': exported_count,
                'expected': len(parametric_table) * len(time_steps)
            })
            
        except Exception as e:
            print(f"\n❌ 파일 처리 실패: {e}")
            import traceback
            traceback.print_exc()
            
            results['failed_files'] += 1
            results['details'].append({
                'file': file_name,
                'status': '❌ Failed',
                'error': str(e)
            })
    
    return results


## 일괄 AEDTPLT Export 실행

file_paths 리스트의 모든 AEDT 파일에 대해:
- Parametric Sweep 결과 (IPeak, PhaseAdvance)
- 각 Time Step
- 모든 조합의 Field Plot을 `.aedtplt` 파일로 export합니다.

In [10]:
"""
일괄 AEDTPLT Export 실행

설정:
- output_base_dir: Export 파일을 저장할 기본 디렉토리
- file_paths: 처리할 AEDT 파일 목록 (이미 정의된 변수 사용)
- setup_name: Transient Setup 이름
- parametric_setup_name: Parametric Setup 이름
- quantity: Export할 필드 물리량 ("A_Vector")
- plot_name: Field Plot 이름
- start_idx: 시작 파일 인덱스 (0부터 시작)
- end_idx: 종료 파일 인덱스 (None이면 끝까지)
"""

import time
from pathlib import Path

# ===== 설정 =====
OUTPUT_BASE_DIR = r"C:\e10_DOE\AEDTPLT_Exports"  # 필요에 따라 수정
SETUP_NAME = "Setup1"
PARAMETRIC_SETUP_NAME = "ParametricSetup1"
QUANTITY = "A_Vector"  # A_Vector 사용
PLOT_NAME = "A_Vector1"

# ===== 파일 범위 설정 =====
START_IDX = 148      # 시작 인덱스 (0부터 시작, 첫 번째 파일)
END_IDX = 149       # 종료 인덱스 (None이면 끝까지, 예: 10이면 0~9번까지 10개 파일)

# file_paths가 정의되어 있는지 확인
if 'file_paths' not in locals():
    print("❌ file_paths 변수가 정의되지 않았습니다.")
    print("이전 셀에서 AEDT 파일 목록을 먼저 가져와주세요.")
else:
    # 파일 범위 선택
    if END_IDX is None:
        selected_files = file_paths[START_IDX:]
        range_text = f"{START_IDX}번 ~ 끝 ({len(selected_files)}개)"
    else:
        selected_files = file_paths[START_IDX:END_IDX]
        range_text = f"{START_IDX}번 ~ {END_IDX-1}번 ({len(selected_files)}개)"
    
    print("=" * 80)
    print("🚀 일괄 AEDTPLT Export 시작")
    print("=" * 80)
    print(f"📁 출력 디렉토리: {OUTPUT_BASE_DIR}")
    print(f"📂 전체 파일 수: {len(file_paths)}")
    print(f"📌 처리 범위: {range_text}")
    print(f"⚙️  Setup: {SETUP_NAME}")
    print(f"📊 Parametric Setup: {PARAMETRIC_SETUP_NAME}")
    print(f"🎨 Field Quantity: {QUANTITY}")
    print(f"🖼️  Plot Name: {PLOT_NAME}")
    print("=" * 80)
    
    # 시작 시간 기록
    start_time = time.perf_counter()
    
    # Export 실행
    results = export_aedtplt_batch(
        m2d_obj=m2d,
        output_base_dir=OUTPUT_BASE_DIR,
        file_paths=selected_files,
        setup_name=SETUP_NAME,
        parametric_setup_name=PARAMETRIC_SETUP_NAME,
        quantity=QUANTITY,
        plot_name=PLOT_NAME,
        create_plot_if_missing=True
    )
    
    # 소요 시간 계산
    elapsed_time = time.perf_counter() - start_time
    
    # ===== 결과 요약 출력 =====
    print("\n" + "=" * 80)
    print("📊 일괄 Export 결과 요약")
    print("=" * 80)
    print(f"⏱️  총 소요 시간: {elapsed_time:.1f} 초 ({elapsed_time/60:.1f} 분)")
    print(f"📦 처리된 파일: {results['processed_files']} / {results['total_files']}")
    print(f"❌ 실패한 파일: {results['failed_files']}")
    print(f"💾 Export된 파일: {results['total_exported']}개")
    print("=" * 80)
    
    # 상세 결과를 DataFrame으로 표시
    if results['details']:
        df_results = pd.DataFrame(results['details'])
        print("\n📋 파일별 처리 결과:")
        display(df_results)
        
        # 성공/실패 통계
        success_count = sum('✅' in str(d['status']) for d in results['details'])
        warning_count = sum('⚠️' in str(d['status']) for d in results['details'])
        failed_count = sum('❌' in str(d['status']) for d in results['details'])
        
        print(f"\n📈 상태별 통계:")
        print(f"  ✅ 성공: {success_count}개")
        print(f"  ⚠️ 경고: {warning_count}개")
        print(f"  ❌ 실패: {failed_count}개")
    
    print("\n" + "=" * 80)
    print("✨ Export 작업 완료!")
    print("=" * 80)

🚀 일괄 AEDTPLT Export 시작
📁 출력 디렉토리: C:\e10_DOE\AEDTPLT_Exports
📂 전체 파일 수: 300
📌 처리 범위: 148번 ~ 148번 (1개)
⚙️  Setup: Setup1
📊 Parametric Setup: ParametricSetup1
🎨 Field Quantity: A_Vector
🖼️  Plot Name: A_Vector1

[1/1] 📁 e10_DOE.aedt
  📂 프로젝트 전환 중...
PyAEDT INFO: Closing the AEDT Project e10_DOE
PyAEDT INFO: Project e10_DOE closed correctly
PyAEDT INFO: Python version 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 20948!
PyAEDT INFO: Project e10_DOE closed correctly
PyAEDT INFO: Python version 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.18.1.
PyAEDT INFO: Returning found Desktop session with PID 20948!
PyAEDT INFO: Project e10_DOE set to active.
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Active Design set to Motor-CAD e10_tutorial
PyAEDT INFO: Project e10_DOE set to ac

Exception ignored on calling ctypes callback function: <bound method AEDT.CreateAedtBlockObj of <ansys.aedt.core.internal.grpc_plugin_dll_class.AEDT object at 0x00000267255033B0>>
Traceback (most recent call last):
  File "c:\Users\EM221\.ansys_python_venvs\pyAEDT_312\Lib\site-packages\ansys\aedt\core\internal\grpc_plugin_dll_class.py", line 416, in CreateAedtBlockObj
    toks = list_in[0].split(":")
           ^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt: 


: 

## 테스트: 단일 파일 Export

전체 파일 처리 전에 단일 파일로 테스트합니다.

In [8]:
# 단일 파일 테스트
if 'file_paths' in locals() and len(file_paths) > 0:
    print("=" * 70)
    print("🧪 단일 파일 테스트 Export")
    print("=" * 70)
    
    # 첫 번째 파일만 선택
    test_file = [file_paths[149]]
    
    print(f"📁 테스트 파일: {Path(test_file[0]).name}")
    print("=" * 70)
    
    # Export 실행
    test_results = export_aedtplt_batch(
        m2d_obj=m2d,
        output_base_dir=r"C:\e10_DOE\AEDTPLT_Exports_Test",
        file_paths=test_file,
        setup_name="Setup1",
        parametric_setup_name="ParametricSetup1",
        quantity="A_Vector",
        plot_name="A_Vector1",
        create_plot_if_missing=True
    )
    
    # 결과 출력
    print("\n" + "=" * 70)
    print("📊 테스트 결과")
    print("=" * 70)
    if test_results['details']:
        df_test = pd.DataFrame(test_results['details'])
        display(df_test)
        print(f"\n💾 Export된 파일 수: {test_results['total_exported']}")
    print("=" * 70)
else:
    print("❌ file_paths 변수가 정의되지 않았거나 비어있습니다.")

  ❌ Export 실패 (Var2, Time44): Failed to execute gRPC AEDT command: ExportFieldPlot
  ❌ Export 실패 (Var2, Time45): Failed to execute gRPC AEDT command: ExportFieldPlot
  ❌ Export 실패 (Var2, Time45): Failed to execute gRPC AEDT command: ExportFieldPlot


KeyboardInterrupt: 

## 📖 사용 가이드

### Export 파일 구조

```
C:\e10_DOE\AEDTPLT_Exports\
├── e10_DOE/
│   ├── e10_DOE_IPeak10A_Phase0deg_Time000.aedtplt
│   ├── e10_DOE_IPeak10A_Phase0deg_Time001.aedtplt
│   ├── e10_DOE_IPeak10A_Phase18deg_Time000.aedtplt
│   ├── e10_DOE_IPeak170.13A_Phase18deg_Time000.aedtplt
│   └── ...
├── Design0002/
│   └── ...
└── Design0003/
    └── ...
```

### 파일명 구조
`{AEDT파일명}_IPeak{값}_Phase{값}_Time{인덱스}.aedtplt`

### 주요 함수

#### 1. `get_parametric_sweep_table(m2d_obj, setup_name)`
- Parametric Sweep 결과 CSV에서 IPeak, PhaseAdvance 값 추출
- 반환: DataFrame with columns ['IPeak', 'PhaseAdvance']

#### 2. `get_time_steps(m2d_obj, setup_name)`
- Transient 해석의 Time Step 목록 추출
- 반환: List of time values (예: ["0.00011494s", "0.00022988s", ...])

#### 3. `export_aedtplt_batch(...)`
- 여러 AEDT 파일에 대해 일괄 Export
- Parametric Sweep 각 Variation × Time Step 조합 처리
- 자동 Field Plot 생성 옵션 제공

### 처리 순서

1. **프로젝트 로드**: 각 AEDT 파일을 순차적으로 열기
2. **Parametric Table 추출**: CSV 파일에서 Sweep 결과 읽기
3. **Time Steps 추출**: Transient 해석 결과에서 시간 목록 가져오기
4. **Field Plot 확인/생성**: B_Vector1 Plot 존재 여부 확인
5. **Export 실행**: 
   - 각 Variation에 대해 IPeak, PhaseAdvance 변수 설정
   - 각 Time Step에 대해 Solution Context 설정
   - AEDTPLT 파일 Export

### 예상 처리 시간

- **단일 Variation × 단일 Time**: ~0.5-1초
- **30 Variations × 100 Time Steps**: ~50-100분 (파일당)
- **전체 150개 파일**: 상당한 시간 소요 예상

### 주의사항

⚠️ **대용량 처리 작업**
- Export 파일이 매우 많이 생성됩니다 (수만 개 이상)
- 디스크 공간 충분히 확보 필요
- 중간에 중단되어도 이미 처리된 파일은 유지됨

⚠️ **AEDT 메모리 사용**
- 장시간 실행 시 AEDT 메모리 누수 가능
- 주기적으로 재시작 권장

⚠️ **파일 잠금**
- `.aedt.lock` 파일로 인한 오류 발생 가능
- 사전에 Lock 파일 제거 (`remove_lock_files()` 함수 사용)